In [1]:
import os

if os.path.exists("AZ-900.pdf"):
    print("The file AZ-900.pdf exists.")

The file AZ-900.pdf exists.


In [14]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import OpenAIEmbeddings 
 

STEP-0 LOAD THE PDF INTO TEXT FORMAT

In [11]:
text_data=PyPDFLoader("AZ-900.pdf").load()
text_data

# Changing metadata of the pdf
for page in text_data:
    page.metadata["author"]="Ruchika Yadav"

text_data


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-26T20:30:23+05:30', 'author': 'Ruchika Yadav', 'moddate': '2026-06-26T20:30:23+05:30', 'title': 'Azure AZ-900 — Study Notes', 'source': 'AZ-900.pdf', 'total_pages': 41, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-26T20:30:23+05:30', 'author': 'Ruchika Yadav', 'moddate': '2026-06-26T20:30:23+05:30', 'title': 'Azure AZ-900 — Study Notes', 'source': 'AZ-900.pdf', 'total_pages': 41, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-26T20:30:23+05:30', 'author': 'Ruchika Yadav', 'moddate': '2026-06-26T20:30:23+05:30', 'title': 'Azure AZ-900 — Study Notes', 'source': 'AZ-900.pdf', 'total_pages': 41, 'page': 2, 'page_label': '3'}, page_content=''),
 Document(metadata={'producer': 'M

CREATING CHUNKS OF THE TEXT DATA

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=10)
chunks=splitter.split_documents(text_data)
chunks
 

[]

CREATION OF THE EMBEDDINGS AND STORING THEM IN VECTOR DATABASE 

In [ ]:
embed_model=OpenAIEmbeddings(model="text-embedding-3-small")

embedded_chunks=embed_model.embed_documents([chunk.page_content for chunk in chunks])
embedded_chunks[0]

In [ ]:
from langchain_community.vectorstores import Chroma

chroma_db=Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

CONNECTION AND RETRIEVAL 

In [ ]:
chroma_db_con=Chroma(persist_directory="./chroma_db", embedding_function=embed_model)   

In [ ]:
chroma_db_con.similarity_search("What is Azure?", k=3)


LLM INTEGREATION AND ASNWER GENERATION 

In [ ]:
llm=ChatOpenAI(model="gpt-4o", temperature=0.1)
llm.invoke("What is Azure?")
  

In [ ]:
user_query=input("Please enter your query: ")
rel_chunks=chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content=[]
for i,chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content=str(rel_chunks_content)

llm.invoke(f"Answer the following question based on the context provided. If the answer is not present in the context, respond with 'I don't know'.\n\nContext: {rel_chunks_content}\n\nQuestion: {user_query}")

user_query=